# Predicting Student Test Scores
## Score: 8.72857

In [1]:
import subprocess
import sys

try:
    import lightgbm as lgb
    from catboost import CatBoostRegressor
    import xgboost as xgb
    from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
    from sklearn.linear_model import Ridge, Lasso, ElasticNet
    from sklearn.cluster import KMeans
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm", "catboost", "xgboost", "--quiet"])
    import lightgbm as lgb
    from catboost import CatBoostRegressor
    import xgboost as xgb
    from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
    from sklearn.linear_model import Ridge, Lasso, ElasticNet
    from sklearn.cluster import KMeans

import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error


In [2]:
train = pd.read_csv('playground-series-s6e1/train.csv')
test = pd.read_csv('playground-series-s6e1/test.csv')
test_ids = test['id'].copy()


In [3]:
def create_features(df, train_df=None):
    df = df.copy()
    
    df['internet_access'] = (df['internet_access'] == 'yes').astype(int)
    
    sleep_quality_map = {'poor': 0, 'average': 1, 'good': 2}
    facility_rating_map = {'low': 0, 'medium': 1, 'high': 2}
    exam_difficulty_map = {'easy': 0, 'moderate': 1, 'hard': 2}
    
    df['sleep_quality_ord'] = df['sleep_quality'].map(sleep_quality_map)
    df['facility_rating_ord'] = df['facility_rating'].map(facility_rating_map)
    df['exam_difficulty_ord'] = df['exam_difficulty'].map(exam_difficulty_map)
    
    df['study_efficiency'] = df['study_hours'] * df['class_attendance']
    df['study_sleep_ratio'] = df['study_hours'] / (df['sleep_hours'] + 1e-5)
    df['attendance_facility'] = df['class_attendance'] * df['facility_rating_ord']
    df['sleep_quality_score'] = df['sleep_hours'] * df['sleep_quality_ord']
    df['study_hours_sq'] = df['study_hours'] ** 2
    df['attendance_sq'] = df['class_attendance'] ** 2
    df['study_per_age'] = df['study_hours'] / (df['age'] + 1e-5)
    df['attendance_per_age'] = df['class_attendance'] / (df['age'] + 1e-5)
    
    df['study_hours_cubed'] = df['study_hours'] ** 3
    df['study_attendance_sleep'] = df['study_hours'] * df['class_attendance'] * df['sleep_hours']
    df['efficiency_sleep'] = df['study_efficiency'] * df['sleep_hours']
    df['study_facility'] = df['study_hours'] * df['facility_rating_ord']
    df['difficulty_facility'] = df['exam_difficulty_ord'] * df['facility_rating_ord']
    df['study_attendance_facility'] = df['study_hours'] * df['class_attendance'] * df['facility_rating_ord']
    df['sleep_attendance'] = df['sleep_hours'] * df['class_attendance']
    df['study_difficulty'] = df['study_hours'] * df['exam_difficulty_ord']
    df['attendance_difficulty'] = df['class_attendance'] * df['exam_difficulty_ord']
    df['total_effort'] = df['study_hours'] + df['class_attendance'] / 10
    df['sleep_ratio_sq'] = df['study_sleep_ratio'] ** 2
    df['study_attendance_ratio'] = df['study_hours'] / (df['class_attendance'] + 1e-5)
    df['efficiency_per_sleep'] = df['study_efficiency'] / (df['sleep_hours'] + 1e-5)
    df['study_facility_difficulty'] = df['study_hours'] * df['facility_rating_ord'] * df['exam_difficulty_ord']
    df['attendance_sleep_quality'] = df['class_attendance'] * df['sleep_hours'] * df['sleep_quality_ord']
    df['study_hours_log'] = np.log1p(df['study_hours'])
    df['attendance_log'] = np.log1p(df['class_attendance'])
    df['sleep_hours_log'] = np.log1p(df['sleep_hours'])
    df['age_squared'] = df['age'] ** 2
    
    df['study_hours_bin'] = pd.cut(df['study_hours'], bins=5, labels=False, duplicates='drop')
    df['attendance_bin'] = pd.cut(df['class_attendance'], bins=5, labels=False, duplicates='drop')
    df['age_group'] = pd.cut(df['age'], bins=[0, 18, 20, 22, 25], labels=[0, 1, 2, 3], duplicates='drop')
    df['age_group'] = df['age_group'].fillna(2).astype(int)
    
    if train_df is not None:
        numeric_cols = ['study_hours', 'class_attendance', 'sleep_hours', 'age']
        cat_cols = ['course', 'study_method', 'gender']
        
        for cat_col in cat_cols:
            for num_col in numeric_cols:
                stats = train_df.groupby(cat_col)[num_col].agg(['mean', 'std', 'min', 'max'])
                df[f'{num_col}_mean_by_{cat_col}'] = df[cat_col].map(stats['mean'])
                df[f'{num_col}_std_by_{cat_col}'] = df[cat_col].map(stats['std'])
                df[f'{num_col}_min_by_{cat_col}'] = df[cat_col].map(stats['min'])
                df[f'{num_col}_max_by_{cat_col}'] = df[cat_col].map(stats['max'])
                df[f'{num_col}_diff_from_mean_{cat_col}'] = df[num_col] - df[f'{num_col}_mean_by_{cat_col}']
        
        cluster_features = ['study_hours', 'class_attendance', 'sleep_hours', 'age']
        kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
        kmeans.fit(train_df[cluster_features])
        df['cluster'] = kmeans.predict(df[cluster_features])
        df['cluster_dist_0'] = np.linalg.norm(df[cluster_features].values - kmeans.cluster_centers_[0], axis=1)
        df['cluster_dist_1'] = np.linalg.norm(df[cluster_features].values - kmeans.cluster_centers_[1], axis=1)
        df['cluster_dist_2'] = np.linalg.norm(df[cluster_features].values - kmeans.cluster_centers_[2], axis=1)
    
    return df

train = create_features(train, train_df=train)
test = create_features(test, train_df=train)


c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python3

In [4]:
categorical_cols = ['gender', 'course', 'sleep_quality', 'study_method']
target_col = 'exam_score'

for col in categorical_cols:
    train[f'{col}_freq'] = train.groupby(col)[col].transform('count')
    test[f'{col}_freq'] = test[col].map(train.groupby(col)[col].count())

kf = KFold(n_splits=5, shuffle=True, random_state=42)

global_mean = train[target_col].mean()
smoothing = 8.0

for col in categorical_cols:
    train[f'{col}_target'] = 0.0
    test[f'{col}_target'] = 0.0
    train[f'{col}_target_std'] = 0.0
    test[f'{col}_target_std'] = 0.0
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
        train_fold = train.iloc[train_idx]
        val_fold = train.iloc[val_idx]
        
        mean_target = train_fold.groupby(col)[target_col].mean()
        std_target = train_fold.groupby(col)[target_col].std()
        count = train_fold.groupby(col)[target_col].count()
        smoothed = (mean_target * count + global_mean * smoothing) / (count + smoothing)
        
        train.loc[val_idx, f'{col}_target'] = val_fold[col].map(smoothed).astype(float)
        train.loc[val_idx, f'{col}_target_std'] = val_fold[col].map(std_target).fillna(0).astype(float)
        
        test_mean = test[col].map(train_fold.groupby(col)[target_col].mean())
        test_std = test[col].map(train_fold.groupby(col)[target_col].std())
        test_count = test[col].map(train_fold.groupby(col)[target_col].count()).fillna(0)
        test_smoothed = (test_mean * test_count + global_mean * smoothing) / (test_count + smoothing)
        test[f'{col}_target'] += test_smoothed.fillna(global_mean).astype(float) / 5
        test[f'{col}_target_std'] += test_std.fillna(0).astype(float) / 5

numeric_cols_to_cap = ['study_hours', 'class_attendance', 'sleep_hours', 'age']
for col in numeric_cols_to_cap:
    q1 = train[col].quantile(0.01)
    q99 = train[col].quantile(0.99)
    train[col] = train[col].clip(lower=q1, upper=q99)
    test[col] = test[col].clip(lower=q1, upper=q99)

drop_cols = ['id', 'exam_score', 'gender', 'course', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']
X = train.drop(drop_cols, axis=1)
y = train[target_col]
X_test = test.drop(['id', 'gender', 'course', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty'], axis=1)


C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_23236\4056137827.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[f'{col}_freq'] = train.groupby(col)[col].transform('count')
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_23236\4056137827.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test[f'{col}_freq'] = test[col].map(train.groupby(col)[col].count())
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_23236\4056137827.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)


In [6]:
seeds = [42, 123]

xgb_params_base = {
    'n_estimators': 2000,
    'learning_rate': 0.011,
    'max_depth': 9,
    'subsample': 0.75,
    'colsample_bytree': 0.75,
    'reg_alpha': 0.12,
    'reg_lambda': 0.12,
    'tree_method': 'hist',
    'device': 'cpu'
}

lgb_params_base = {
    'n_estimators': 2000,
    'learning_rate': 0.011,
    'max_depth': 9,
    'subsample': 0.75,
    'colsample_bytree': 0.75,
    'reg_alpha': 0.12,
    'reg_lambda': 0.12,
    'verbose': -1
}

cat_params_base = {
    'iterations': 2000,
    'learning_rate': 0.011,
    'depth': 9,
    'subsample': 0.75,
    'colsample_bylevel': 0.75,
    'l2_leaf_reg': 0.12,
    'verbose': False
}


In [7]:
all_oof = []
all_test_preds = []
model_names = []

for seed in seeds:
    xgb_params = {**xgb_params_base, 'random_state': seed}
    lgb_params = {**lgb_params_base, 'random_state': seed}
    cat_params = {**cat_params_base, 'random_seed': seed}
    
    xgb_oof = np.zeros(len(train))
    lgb_oof = np.zeros(len(train))
    cat_oof = np.zeros(len(train))
    
    xgb_test_preds = np.zeros(len(test))
    lgb_test_preds = np.zeros(len(test))
    cat_test_preds = np.zeros(len(test))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        
        xgb_model = xgb.XGBRegressor(**xgb_params)
        xgb_model.fit(X_train_fold, y_train_fold, 
                      eval_set=[(X_val_fold, y_val_fold)],
                      verbose=False)
        xgb_oof[val_idx] = xgb_model.predict(X_val_fold)
        xgb_test_preds += xgb_model.predict(X_test) / 5
        
        lgb_model = lgb.LGBMRegressor(**lgb_params)
        lgb_model.fit(X_train_fold, y_train_fold,
                      eval_set=[(X_val_fold, y_val_fold)],
                      callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
        lgb_oof[val_idx] = lgb_model.predict(X_val_fold)
        lgb_test_preds += lgb_model.predict(X_test) / 5
        
        cat_model = CatBoostRegressor(**cat_params)
        cat_model.fit(X_train_fold, y_train_fold,
                      eval_set=(X_val_fold, y_val_fold),
                      early_stopping_rounds=150)
        cat_oof[val_idx] = cat_model.predict(X_val_fold)
        cat_test_preds += cat_model.predict(X_test) / 5
    
    all_oof.append(xgb_oof)
    all_oof.append(lgb_oof)
    all_oof.append(cat_oof)
    all_test_preds.append(xgb_test_preds)
    all_test_preds.append(lgb_test_preds)
    all_test_preds.append(cat_test_preds)
    model_names.extend([f'xgb_{seed}', f'lgb_{seed}', f'cat_{seed}'])
    
    xgb_rmse = np.sqrt(mean_squared_error(y, xgb_oof))
    lgb_rmse = np.sqrt(mean_squared_error(y, lgb_oof))
    cat_rmse = np.sqrt(mean_squared_error(y, cat_oof))
    print(f'Seed {seed} - XGB: {xgb_rmse:.5f}, LGB: {lgb_rmse:.5f}, Cat: {cat_rmse:.5f}')

all_oof = np.column_stack(all_oof)
all_test_preds = np.column_stack(all_test_preds)


Training until validation scores don't improve for 150 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's l2: 77.1253
Training until validation scores don't improve for 150 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's l2: 77.3404
Training until validation scores don't improve for 150 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's l2: 77.1339
Training until validation scores don't improve for 150 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's l2: 77.4513
Training until validation scores don't improve for 150 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's l2: 77.706
Seed 42 - XGB: 8.76874, LGB: 8.79496, Cat: 8.80231
Training until validation scores don't improve for 150 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's l2: 77.0996
Training until validation scores don't improve for 150 rounds
Did not meet early stopping. Best iteration is:
[2000]

In [8]:
stacking_oof = np.zeros(len(train))
stacking_test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_stack_train = all_oof[train_idx]
    X_stack_val = all_oof[val_idx]
    y_stack_train = y.iloc[train_idx]
    
    ridge = Ridge(alpha=10.0, random_state=42)
    lasso = Lasso(alpha=1.0, random_state=42, max_iter=2000)
    elastic = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42, max_iter=2000)
    xgb_meta = xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4, 
                                random_state=42, tree_method='hist')
    lgb_meta = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                                random_state=42, verbose=-1)
    
    ridge.fit(X_stack_train, y_stack_train)
    lasso.fit(X_stack_train, y_stack_train)
    elastic.fit(X_stack_train, y_stack_train)
    xgb_meta.fit(X_stack_train, y_stack_train, 
                 eval_set=[(X_stack_val, y.iloc[val_idx])], verbose=False)
    lgb_meta.fit(X_stack_train, y_stack_train,
                 eval_set=[(X_stack_val, y.iloc[val_idx])],
                 callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    
    ridge_pred = ridge.predict(X_stack_val)
    lasso_pred = lasso.predict(X_stack_val)
    elastic_pred = elastic.predict(X_stack_val)
    xgb_meta_pred = xgb_meta.predict(X_stack_val)
    lgb_meta_pred = lgb_meta.predict(X_stack_val)
    
    meta_preds = np.column_stack([ridge_pred, lasso_pred, elastic_pred, xgb_meta_pred, lgb_meta_pred])
    meta_rmses = [np.sqrt(mean_squared_error(y.iloc[val_idx], pred)) for pred in meta_preds.T]
    meta_weights = np.array([1/rmse for rmse in meta_rmses])
    meta_weights = meta_weights / meta_weights.sum()
    
    stacking_oof[val_idx] = np.average(meta_preds, axis=1, weights=meta_weights)
    
    test_meta_preds = np.column_stack([
        ridge.predict(all_test_preds),
        lasso.predict(all_test_preds),
        elastic.predict(all_test_preds),
        xgb_meta.predict(all_test_preds),
        lgb_meta.predict(all_test_preds)
    ])
    stacking_test_preds += np.average(test_meta_preds, axis=1, weights=meta_weights) / 5

stacking_rmse = np.sqrt(mean_squared_error(y, stacking_oof))
print(f'Stacking OOF RMSE: {stacking_rmse:.5f}')


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.029e+04, tolerance: 1.806e+04
  model = cd_fast.enet_coordinate_descent(


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[116]	valid_0's l2: 76.6797


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.946e+04, tolerance: 1.804e+04
  model = cd_fast.enet_coordinate_descent(


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[126]	valid_0's l2: 76.7688


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.841e+04, tolerance: 1.804e+04
  model = cd_fast.enet_coordinate_descent(


...

C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.886e+04, tolerance: 1.802e+04
  model = cd_fast.enet_coordinate_descent(


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[179]	valid_0's l2: 77.2525


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Stacking OOF RMSE: 8.76433


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [9]:
rmses = [np.sqrt(mean_squared_error(y, all_oof[:, i])) for i in range(all_oof.shape[1])]
weights = np.array([1/rmse for rmse in rmses])
weights = weights / weights.sum()

simple_ensemble = np.average(all_test_preds, axis=1, weights=weights)

ratios = np.arange(0.6, 0.95, 0.01)
best_ratio = 0.8
best_rmse = float('inf')

simple_ensemble_oof = np.average(all_oof, axis=1, weights=weights)
for ratio in ratios:
    test_oof = ratio * stacking_oof + (1 - ratio) * simple_ensemble_oof
    test_rmse = np.sqrt(mean_squared_error(y, test_oof))
    if test_rmse < best_rmse:
        best_rmse = test_rmse
        best_ratio = ratio

print(f'Best stacking ratio: {best_ratio:.2f} (RMSE: {best_rmse:.5f})')

final_predictions = best_ratio * stacking_test_preds + (1 - best_ratio) * simple_ensemble

submission = pd.DataFrame({
    'id': test_ids,
    'exam_score': final_predictions
})
submission.to_csv('submission.csv', index=False)


Best stacking ratio: 0.94 (RMSE: 8.76443)


In [10]:
final_oof = best_ratio * stacking_oof + (1 - best_ratio) * simple_ensemble_oof
final_rmse = np.sqrt(mean_squared_error(y, final_oof))
print(f'Final Ensemble OOF RMSE: {final_rmse:.5f}')


Final Ensemble OOF RMSE: 8.76443


In [11]:
import winsound
import time
notes = [(523, 200), (659, 200), (784, 200), (1047, 400), (784, 200), (1047, 600)]
for freq, dur in notes:
    winsound.Beep(freq, dur)
    time.sleep(0.05)
